# Week 2 First Engine Test in Google Colab

Play chess against Stockfish with **nothing installed on your computer** — everything runs in this notebook.

**How to use it:** run the cells top to bottom with the ▶ button (or `Shift+Enter`). Step 4 is the game itself — you re-run that one cell for every move you make.

> ⚠️ Colab wipes its files when the session disconnects. Step 7 downloads your games — do it before you leave.



In [ ]:
#@title Step 1 · Install everything (≈30 seconds)
!pip install chess -q
!apt-get -qq install -y stockfish > /dev/null

import chess
ENGINE_PATH = "/usr/games/stockfish"   # where apt puts it on Colab's Ubuntu
print("python-chess", chess.__version__, "ready · engine at", ENGINE_PATH)

**What just happened:** `chess` (the python-chess library) knows the rules and speaks UCI — the universal engine protocol. `stockfish` is the engine itself, a separate program. Colab's copy comes from Ubuntu's package archive, so it's a version or two behind the newest release — plenty for us.

In [ ]:
#@title Step 2 · Start the engine and set a fair strength
import chess.engine, chess.svg, chess.pgn
from IPython.display import display, SVG, clear_output

TARGET_ELO = 1350  #@param {type:"slider", min:1000, max:3200, step:50}

engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)

# Ask the engine what strength range IT supports, and clamp our target to it.
opt = engine.options["UCI_Elo"]
ELO = max(opt.min, min(opt.max, TARGET_ELO))
engine.configure({"UCI_LimitStrength": True, "UCI_Elo": ELO})

print(f"Opponent: {engine.id['name']}")
print(f"This engine's UCI_Elo range is {opt.min}–{opt.max} → playing at {ELO}")

def show(board, last=None):
    arrows = [chess.svg.Arrow(last.from_square, last.to_square)] if last else []
    display(SVG(chess.svg.board(board, size=360, lastmove=last, arrows=arrows)))

**Why the clamp?** `UCI_Elo` is ignored unless `UCI_LimitStrength` is `True`, and every engine advertises its own supported range (this Stockfish's floor is around 1320 — it literally cannot pretend to be weaker). Asking the engine instead of hard-coding numbers is a habit worth keeping.

In [ ]:
#@title Step 3 · New game (run again anytime to restart)
board = chess.Board()
show(board)
print("You are White. Go to Step 4 to make your first move.")

## Step 4 — this cell *is* the game

Type your move in the box on the right (normal notation: `e4`, `Nf3`, `O-O` — or coordinates like `e2e4`), then run the cell. Stockfish answers and the board redraws with its move arrowed. **Run this cell once per move.**

In [ ]:
#@title Step 4 · Your move — edit the box, then run this cell each turn
your_move = "O-O"  #@param {type:"string"}

def parse(board, text):
    try:
        return board.parse_san(text.strip())
    except ValueError:
        pass
    try:
        m = chess.Move.from_uci(text.strip().lower())
        return m if m in board.legal_moves else None
    except ValueError:
        return None

if board.is_game_over():
    print("Game over:", board.result(), "— run Step 3 for a new game.")
else:
    m = parse(board, your_move)
    if m is None:
        some = ", ".join(board.san(x) for x in list(board.legal_moves)[:8])
        print(f"'{your_move}' isn't legal here. Some legal moves: {some} …")
    else:
        san_you = board.san(m)
        board.push(m)
        if board.is_game_over():
            show(board, m)
            print(f"You: {san_you} — game over: {board.result()}")
        else:
            reply = engine.play(board, chess.engine.Limit(time=0.5)).move
            san_eng = board.san(reply)
            board.push(reply)
            show(board, reply)
            print(f"You: {san_you}   ·   Stockfish: {san_eng}")
            if board.is_check():
                print("Check!")

In [ ]:
#@title Step 5 (optional) · Whole game in one cell — type moves as prompted
# Colab supports input(): a text box appears under the cell while this runs.
# Moves like e4 / Nf3 / O-O · commands: eval, undo, resign
board = chess.Board()
while not board.is_game_over():
    clear_output(wait=True)
    show(board)
    text = input("Your move (or eval / undo / resign): ").strip()
    low = text.lower()
    if low == "resign":
        print("You resigned.")
        break
    if low == "undo":
        if len(board.move_stack) >= 2:
            board.pop(); board.pop()
        continue
    if low == "eval":
        info = engine.analyse(board, chess.engine.Limit(time=0.3))
        input(f"Eval (White's view): {info['score'].white()} — press Enter")
        continue
    m = parse(board, text)
    if m is None:
        input(f"'{text}' isn't legal — press Enter and try again")
        continue
    board.push(m)
    if board.is_game_over():
        break
    board.push(engine.play(board, chess.engine.Limit(time=0.5)).move)
clear_output(wait=True)
show(board)
print("Result:", board.result() if board.is_game_over() else "unfinished")

In [ ]:
#@title Step 6 · Coach's corner — evaluation and the engine's suggestion
info = engine.analyse(board, chess.engine.Limit(time=0.5))
print("Evaluation (White's point of view):", info["score"].white())
hint = info.get("pv", [None])[0]
if hint:
    print("What the engine would play here:", board.san(hint))
    display(SVG(chess.svg.board(board, size=360,
        arrows=[chess.svg.Arrow(hint.from_square, hint.to_square, color="#9A6B1E")])))

In [ ]:
#@title Step 7 · Save this game as PGN and download it
import datetime
game = chess.pgn.Game.from_board(board)
game.headers["Event"] = "Gambit Lab Colab game"
game.headers["Date"]  = datetime.date.today().strftime("%Y.%m.%d")
game.headers["White"] = "Human"
game.headers["Black"] = f"Stockfish ~{ELO}"
with open("my_games.pgn", "a", encoding="utf-8") as f:
    print(game, file=f, end="\n\n")
print(game)
try:
    from google.colab import files
    files.download("my_games.pgn")   # session storage is temporary — keep a copy!
except ImportError:
    print("\n(Not running in Colab — my_games.pgn saved beside this notebook.)")

Every game you save here is in exactly the format the Chess Blend's analysis pipeline ingests — practice games become dataset rows for the Almanac.

In [ ]:
#@title Bonus · Watch the weakest setting fight a 2400 (same engine, two personalities)
import time
weak   = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)
strong = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)
weak.configure({"UCI_LimitStrength": True, "UCI_Elo": opt.min})
strong.configure({"UCI_LimitStrength": True, "UCI_Elo": min(opt.max, 2400)})

b = chess.Board()
while not b.is_game_over() and len(b.move_stack) < 120:
    eng = weak if b.turn == chess.WHITE else strong
    mv = eng.play(b, chess.engine.Limit(time=0.1)).move
    b.push(mv)
    clear_output(wait=True)
    display(SVG(chess.svg.board(b, size=360, lastmove=mv)))
    print(f"White = {opt.min} Elo · Black = {min(opt.max, 2400)} Elo · move {b.fullmove_number}")
    time.sleep(0.15)
print("Result:", b.result() if b.is_game_over() else "adjourned at move 60")
weak.quit(); strong.quit()

## Exercises

1. **Read a whole engine.** Run `!git clone https://github.com/thomasahle/sunfish` — a complete chess engine in about 111 lines of pure Python. Read `sunfish.py` end to end; it's the best teaching artifact in computer chess.
2. **Build the coach.** Change Step 4 so that after each of your moves, the engine prints one sentence: strong, fine, or mistake (hint: compare the eval before and after).
3. **Mine your games.** Load `my_games.pgn` with `chess.pgn.read_game()` and find your three biggest eval drops — those are your blunders.
4. **Rematch at your level.** Move the Step 2 slider to ~100 points above your rating and play again. Losing narrowly teaches; losing to a wall doesn't.

## When it breaks

| Symptom | Fix |
|---|---|
| `NameError: engine is not defined` | The runtime restarted. Re-run Steps 1–2. |
| "engine process died" | Same — re-run Step 2 to relaunch Stockfish. |
| My files disappeared | Colab session storage is temporary. Step 7 downloads `my_games.pgn`; run it before closing. |
| Step 4 does nothing new | You edited the move box but didn't re-run the cell — press ▶ again. |
| Want the newest Stockfish | Ubuntu's package is a version or two old. Download the latest official build from github.com/official-stockfish/Stockfish/releases, upload it, `!chmod +x` it, and point `ENGINE_PATH` at it. |

---
